In [2]:
import pandas as pd
import time
import re
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup


# ─────────────────────────────────────────────────────────────
# Monarch lookup
# ─────────────────────────────────────────────────────────────
def get_monarch(year: int) -> str:
    if year <= 1951:
        return "King George VI"
    elif year <= 2022:
        return "Queen Elizabeth II"
    else:
        return "King Charles III"


# ─────────────────────────────────────────────────────────────
# Browser setup
# ─────────────────────────────────────────────────────────────
options = Options()
options.add_argument("--incognito")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)
driver.set_window_size(1400, 900)


# ─────────────────────────────────────────────────────────────
# Cookie handler
# ─────────────────────────────────────────────────────────────
def accept_cookies(driver, timeout=8):
    try:
        WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable(
                (By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
            )
        ).click()
        print("Cookies accepted")
    except:
        pass


# ─────────────────────────────────────────────────────────────
# Remove editorial intro paragraph
# ─────────────────────────────────────────────────────────────
def remove_editorial_intro(paragraphs):
    if not paragraphs:
        return paragraphs

    first = paragraphs[0].lower()

    editorial_patterns = [
        r"\bin (his|her) christmas broadcast",
        r"\bthe (king|queen) said",
        r"\bthe following is the text",
        r"\bthe broadcast was recorded",
        r"\bthis was the first",
        r"\bsummary\b",
        r"\bdelivered on\b",
        r"\baddressed the nation\b"
    ]

    if any(re.search(p, first) for p in editorial_patterns):
        return paragraphs[1:]

    return paragraphs


# ─────────────────────────────────────────────────────────────
# Speech extractor (robust for all layouts)
# ─────────────────────────────────────────────────────────────
def extract_speech_body(driver):

    soup = BeautifulSoup(driver.page_source, "html.parser")

    containers = [
        "div.chapters",
        "div.field--name-body",
        "article",
        "main"
    ]

    for selector in containers:
        container = soup.select_one(selector)
        if not container:
            continue

        for junk in container.select("figure, script, style, nav, header, footer"):
            junk.decompose()

        paragraphs = [
            p.get_text(strip=True)
            for p in container.find_all("p")
            if p.get_text(strip=True)
        ]

        paragraphs = remove_editorial_intro(paragraphs)

        if paragraphs:
            return "\n\n".join(paragraphs)

    return None


# ─────────────────────────────────────────────────────────────
# PHASE 1 — Load search page
# ─────────────────────────────────────────────────────────────
START_URL = "https://www.royal.uk/media-centre/speeches"
driver.get(START_URL)
accept_cookies(driver)
time.sleep(3)

# Oldest possible date
date_input = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "input.bef-datepicker"))
)
date_input.clear()
date_input.send_keys("01/01/1900")
date_input.send_keys(Keys.ENTER)

# Search
search_box = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.ID, "edit-text"))
)
search_box.clear()
search_box.send_keys("Christmas broadcast")
search_box.send_keys(Keys.ENTER)

time.sleep(4)


# ─────────────────────────────────────────────────────────────
# PHASE 2 — Collect URLs (deduplicated early)
# ─────────────────────────────────────────────────────────────
speech_urls = {}

page_num = 1
while True:
    print(f"Scanning page {page_num}")
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(strip=True)

        if (
            href.startswith("/")
            and "christmas" in href.lower()
            and re.search(r"\b(19|20)\d{2}\b", href)
        ):
            full_url = "https://www.royal.uk" + href
            speech_urls[full_url] = title

    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "li.pager__item--next a")
        driver.get(next_btn.get_attribute("href"))
        page_num += 1
    except:
        break

print(f"Collected {len(speech_urls)} candidate pages")


# ─────────────────────────────────────────────────────────────
# PHASE 3 — Scrape speeches (canonical dedupe)
# ─────────────────────────────────────────────────────────────
visited_pages = set()
records = []

for i, (url, title) in enumerate(speech_urls.items(), 1):

    print(f"[{i}/{len(speech_urls)}] {title}")

    try:
        driver.get(url)

        # Canonical URL dedupe (fixes duplicates!)
        canonical = driver.current_url.split("?")[0].rstrip("/")
        if canonical in visited_pages:
            print("   Duplicate alias skipped")
            continue
        visited_pages.add(canonical)

        # Extract year
        year_match = re.search(r"\b(19|20)\d{2}\b", canonical + title)
        if not year_match:
            print("   No year found")
            continue

        year = int(year_match.group())
        monarch = get_monarch(year)
        date = datetime(year, 12, 25)

        # Wait for content
        try:
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "main, article, div.chapters"))
            )
        except:
            pass

        speech = extract_speech_body(driver)

        if not speech or len(speech.split()) < 80:
            print("   Too short — skipped")
            continue

        records.append({
            "date": date.strftime("%Y-%m-%d"),
            "year": year,
            "monarch": monarch,
            "title": title,
            "speech": speech,
            "url": canonical
        })

        print(f"   OK — {len(speech.split())} words")

    except Exception as e:
        print("   Failed:", e)


driver.quit()



Cookies accepted
Scanning page 1
Scanning page 2
Scanning page 3
Scanning page 4
Scanning page 5
Scanning page 6
Scanning page 7
Scanning page 8
Collected 72 candidate pages
[1/72] Read moreabout The King's Christmas Broadcast 2023
   OK — 677 words
[2/72] Read moreabout The King's Christmas Broadcast 2022
   OK — 619 words
[3/72] Read moreabout The Christmas Broadcast 2021
   OK — 702 words
[4/72] Read moreabout The Christmas Broadcast 2020
   OK — 608 words
[5/72] Read moreabout The Queen’s Christmas Broadcast 2019
   Too short — skipped
[6/72] Read moreabout The Queen’s Christmas Broadcast 2018
   Too short — skipped
[7/72] Read moreabout The Christmas Broadcast 2017
   Too short — skipped
[8/72] Read moreabout The Christmas Broadcast 2016
   Too short — skipped
[9/72] Read moreabout Christmas Broadcast 2015
   Too short — skipped
[10/72] Read moreabout Christmas Broadcast 2011
   Too short — skipped
[11/72] Read moreabout Christmas Broadcast 2012
   Too short — skipped
[12/72] Read

In [3]:
# ─────────────────────────────────────────────────────────────
# Save CSV
# ─────────────────────────────────────────────────────────────
df = pd.DataFrame(records).sort_values("year")
df.to_csv("royal_christmas_speeches.csv", index=False)

print("\nDONE")
print(f"Saved {len(df)} speeches")


DONE
Saved 64 speeches
